In [ ]:
"""
Guardrails are controls placed around an LLM or agent to validate inputs, 
retrieved context, tool calls, and outputs. They enforce safety, security, 
correctness, and business rules before allowing the system to proceed.
"""

"""
## Input guardrail

An input guardrail checks the user's request belfore it reaches the main LLM/agent
Guardrails aren't necessarily just keyword filters

In production, you might use:

* rules/regex
* classifiers
* LLM-based classifires
* moderation models
* PII detection
* scehma validation
* authentication checks

## Tool call guardrail

- Suppose your banking agent has tools
- User says: Transfer ₹50,000 to account 123456
- Put a guardrail between the LLM and the tool.

User
 ↓
Agent
 ↓
LLM decides:
transfer_money(50000, 123456)
 ↓
Tool Guardrail
 ↓
 ┌─────────────────────┐
 │ Is this operation   │
 │ authorized?         │
 │ Is amount valid?    │
 │ Is confirmation     │
 │ required?           │
 └──────────┬──────────┘
            ↓
        Execute / Block

## Output guardrail

- Suppose the LLM generates: Your account balance is ₹2,45,000. Your account number is 1234567890.

LLM response
     ↓
Output Guardrail
     ↓
PII / sensitive-data detection
     ↓
 ┌───────────┐
 │ Sensitive?│
 └─────┬─────┘
       │
   Yes │ No
       │
       ↓
 Redact      Return response

- Your account balance is ₹2,45,000. Your account number is ****7890.

## Hallucination guardrail

- This is especially relevant to RAG
- Suppose your retrieved context says: Debit cards can be blocked through the mobile application.

- But LLM says: You can block your debit card through the mobile application or by sending an SMS to 56789.

- The SMS information wasn't in the retrieved context. This is potential groundness.

Retrieved Context
       ↓
      LLM
       ↓
Generated Answer
       ↓
Groundedness Checker
       ↓
Does answer follow the context?
       ↓
   Yes       No
    ↓         ↓
 Return     Regenerate /
 response   refuse
"""


In [1]:
"""
                    USER
                      │
                      ▼
             ┌─────────────────┐
             │ Input Guardrail  │
             │                 │
             │ • PII           │
             │ • Injection     │
             │ • Topic         │
             │ • Abuse         │
             └────────┬────────┘
                      │
                      ▼
                Query Processing
                      │
                      ▼
                  Retriever
                      │
                      ▼
             Retrieval Guardrail
                      │
               relevant chunks?
                 /          \
               No            Yes
               │              │
               ▼              ▼
            Refuse           LLM
                              │
                              ▼
                       Tool decision
                              │
                              ▼
                       Tool Guardrail
                              │
                       authorized?
                         /       \
                       No         Yes
                       │           │
                       ▼           ▼
                    Block       Execute
                                   │
                                   ▼
                              LLM Response
                                   │
                                   ▼
                        Output Guardrail
                                   │
                       ┌───────────┴───────────┐
                       │                       │
                  Safe/Grounded           Unsafe
                       │                       │
                       ▼                       ▼
                    USER                 Redact/Refuse
"""

'\n                    USER\n                      │\n                      ▼\n             ┌─────────────────┐\n             │ Input Guardrail  │\n             │                 │\n             │ • PII           │\n             │ • Injection     │\n             │ • Topic         │\n             │ • Abuse         │\n             └────────┬────────┘\n                      │\n                      ▼\n                Query Processing\n                      │\n                      ▼\n                  Retriever\n                      │\n                      ▼\n             Retrieval Guardrail\n                      │\n               relevant chunks?\n                 /                         No            Yes\n               │              │\n               ▼              ▼\n            Refuse           LLM\n                              │\n                              ▼\n                       Tool decision\n                              │\n                              ▼\n           

In [1]:
"""
Let say we are creating an ingestion pipeline for our 
RAG system what we have to do is while chunking we should be carefull that it 
secrets like, API_Keys, passwords are not chunked and converteded into embedding
and stored into vectordb
"""

import re

text = """
API_KEY=sk-abc123xyz789
password=MySecret@123
"""

text = re.sub(
    r'(?i)(api[_-]?key\s*[=:]\s*)\S+',
    r'\1[REDACTED]',
    text
)

text = re.sub(
    r'(?i)(password\s*[=:]\s*)\S+',
    r'\1[REDACTED]',
    text
)

print(text)


API_KEY=[REDACTED]
password=[REDACTED]



🔐 Data protection

- PII
- Sensitive data
- Secrets
- Data leakage
- Data exfiltration

🧠 LLM attacks

- Prompt injection
- Indirect prompt injection
- Jailbreaking
- Prompt leakage

🤖 Agent security

- Excessive agency
- Broken access control
- Insecure tool use
- Privilege escalation

✅ Reliability

- Hallucination
- Groundedness
- Faithfulness
- Output validation

                    USER
                      ↓
              Input Guardrails
                      ↓
             Authentication
                      ↓
               Authorization
                      ↓
                 Retriever
                      ↓
          ┌─────────────────────┐
          │ Retrieval Guardrails │
          │ • access control     │
          │ • relevance          │
          │ • injection checks   │
          └──────────┬──────────┘
                     ↓
                    LLM
                     ↓
              Tool-call Guardrail
                     ↓
          ┌──────────────────────┐
          │ • authorization      │
          │ • validation         │
          │ • least privilege    │
          │ • human approval     │
          └──────────┬───────────┘
                     ↓
                   Tools
                     ↓
               Output Guardrails
                     ↓
             PII / secrets check
             groundedness check
                     ↓
                   USER